# BookWise — Content-Based Filtering
Menggunakan TF-IDF + Cosine Similarity

In [ ]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import precision_score
import joblib, os

DATA_DIR  = '../../data'
MODEL_DIR = '../model'
os.makedirs(MODEL_DIR, exist_ok=True)

books   = pd.read_csv(f'{DATA_DIR}/BX-Books.csv',        sep=';', encoding='latin-1', on_bad_lines='skip')
ratings = pd.read_csv(f'{DATA_DIR}/BX-Book-Ratings.csv', sep=';', encoding='latin-1', on_bad_lines='skip')
books.columns   = books.columns.str.strip().str.replace('"', '')
ratings.columns = ratings.columns.str.strip().str.replace('"', '')

books = books[['ISBN','Book-Title','Book-Author','Publisher','Image-URL-M']].dropna(subset=['ISBN','Book-Title'])
books.drop_duplicates(subset='ISBN', inplace=True)
print(f'Books loaded: {len(books):,}')

In [ ]:
# Ambil 10k buku paling populer
ratings_explicit = ratings[ratings['Book-Rating'] > 0]
popular_isbns = ratings_explicit['ISBN'].value_counts().head(10000).index
cb_books = books[books['ISBN'].isin(popular_isbns)].reset_index(drop=True)

cb_books['features'] = (cb_books['Book-Title'].fillna('') + ' ' +
                        cb_books['Book-Author'].fillna('') + ' ' +
                        cb_books['Publisher'].fillna(''))

tfidf     = TfidfVectorizer(max_features=5000, stop_words='english')
tfidf_mat = tfidf.fit_transform(cb_books['features'])
print(f'TF-IDF matrix shape: {tfidf_mat.shape}')

In [ ]:
cosine_sim = cosine_similarity(tfidf_mat, tfidf_mat)
isbn_index = {isbn: idx for idx, isbn in enumerate(cb_books['ISBN'])}

def get_recommendations(isbn, top_k=10):
    if isbn not in isbn_index:
        return []
    idx = isbn_index[isbn]
    sim_scores = sorted(enumerate(cosine_sim[idx]), key=lambda x: x[1], reverse=True)[1:top_k+1]
    return cb_books.iloc[[i[0] for i in sim_scores]][['ISBN','Book-Title','Book-Author']]

# Test
sample_isbn = cb_books['ISBN'].iloc[0]
print(f'Rekomendasi untuk: {cb_books[cb_books["ISBN"]==sample_isbn]["Book-Title"].values[0]}')
print(get_recommendations(sample_isbn, 5).to_string(index=False))

In [ ]:
# Evaluasi: Precision@K
# Untuk setiap user, cek apakah buku yang direkomendasikan ada di histori rating tinggi
high_ratings = ratings_explicit[ratings_explicit['Book-Rating'] >= 7]

precisions = []
sample_users = high_ratings['User-ID'].value_counts().head(100).index

for user in sample_users:
    user_books = high_ratings[high_ratings['User-ID'] == user]['ISBN'].tolist()
    if len(user_books) < 2:
        continue
    seed_isbn = user_books[0]
    recs = get_recommendations(seed_isbn, 10)
    if len(recs) == 0:
        continue
    hits = len(set(recs['ISBN'].tolist()) & set(user_books[1:]))
    precisions.append(hits / 10)

print(f'Precision@10 (Content-Based): {np.mean(precisions):.4f}')

# Simpan model
cb_model = {'cosine_sim': cosine_sim, 'isbn_index': isbn_index, 'tfidf': tfidf}
joblib.dump(cb_model, f'{MODEL_DIR}/content_based.pkl')
cb_books.to_pickle(f'{MODEL_DIR}/books.pkl')
print('✅ Model saved.')